# OfficeQA Retrieval-Only Colab Runner

This notebook is the primary Colab workflow for the project. It handles:

1. mounting Google Drive
2. cloning or updating the repo
3. downloading missing Treasury Bulletin PDFs
4. preparing the page manifest and BM25 index
5. building multimodal FAISS indexes for CLIP and SigLIP
6. running all retrieval experiments
7. exporting plots and CSVs for the final report

Use `RUN_MODE = "smoke"` first to validate the pipeline quickly, then switch to `RUN_MODE = "full"` for the final benchmark.


## Before You Run

For `RUN_MODE = "full"`, make sure `officeqa_pro.csv` is already in:

`/content/drive/MyDrive/officeqa/officeqa_pro.csv`

The notebook will download missing PDFs into:

`/content/drive/MyDrive/officeqa/pdfs/`

For `RUN_MODE = "smoke"`, the notebook uses the bundled smoke CSV from the repo, so you do not need to upload a separate smoke file.


In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/hungngo2/cs445-final-project-retrieval.git'
REPO_DIR = Path('/content/officeqa-retrieval')

if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'pip'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[ml,faiss,dev]'], check=True)
subprocess.run(['nvidia-smi'], check=False)

print(f'Repo ready at {REPO_DIR}')


In [ ]:
import os
import shlex
import subprocess
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

RUN_MODE = 'smoke'  # change to 'full' for the real benchmark
RUN_TAG = 'officeqa_smoke_faiss' if RUN_MODE == 'smoke' else 'officeqa_pro_faiss'
TOP_K = 50
BATCH_SIZE = 8
PAGE_BATCH_SIZE = 16 if RUN_MODE == 'smoke' else 32
DPI = 150
FORCE_REBUILD = False
DOWNLOAD_MISSING_PDFS = True

OFFICEQA_ROOT = Path('/content/drive/MyDrive/officeqa')
PDF_DIR = OFFICEQA_ROOT / 'pdfs'
ARTIFACT_DIR = OFFICEQA_ROOT / 'artifacts' / RUN_TAG
RESULTS_DIR = OFFICEQA_ROOT / 'results' / RUN_TAG
RENDER_CACHE = ARTIFACT_DIR / 'render_cache'
ANALYSIS_DIR = RESULTS_DIR / 'analysis'

QUESTIONS_CSV = REPO_DIR / 'data' / 'officeqa_pro_smoke.csv' if RUN_MODE == 'smoke' else OFFICEQA_ROOT / 'officeqa_pro.csv'
if not QUESTIONS_CSV.exists():
    raise FileNotFoundError(f'Missing questions CSV: {QUESTIONS_CSV}')

for path in [PDF_DIR, ARTIFACT_DIR, RESULTS_DIR, ANALYSIS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

BM25_INDEX_PATH = ARTIFACT_DIR / 'page_bm25.pkl'
MANIFEST_PATH = ARTIFACT_DIR / 'page_manifest.jsonl'
SANITY_PATH = ARTIFACT_DIR / 'sanity_questions.json'
SUMMARY_PATH = RESULTS_DIR / 'summary.csv'

EXPERIMENTS = [
    {
        'run_name': 'bm25',
        'eval_model': 'bm25',
        'index_path': BM25_INDEX_PATH,
        'index_model': None,
        'crop_mode': None,
    },
    {
        'run_name': 'clip_faiss_full',
        'eval_model': 'clip_faiss',
        'index_path': ARTIFACT_DIR / 'clip_faiss_full',
        'index_model': 'clip',
        'crop_mode': 'full',
    },
    {
        'run_name': 'clip_faiss_fixed',
        'eval_model': 'clip_faiss',
        'index_path': ARTIFACT_DIR / 'clip_faiss_fixed',
        'index_model': 'clip',
        'crop_mode': 'fixed_2x2',
    },
    {
        'run_name': 'siglip_faiss_full',
        'eval_model': 'siglip_faiss',
        'index_path': ARTIFACT_DIR / 'siglip_faiss_full',
        'index_model': 'siglip',
        'crop_mode': 'full',
    },
    {
        'run_name': 'siglip_faiss_fixed',
        'eval_model': 'siglip_faiss',
        'index_path': ARTIFACT_DIR / 'siglip_faiss_fixed',
        'index_model': 'siglip',
        'crop_mode': 'fixed_2x2',
    },
]


def run_cmd(args, cwd=REPO_DIR, env=None):
    args = [str(arg) for arg in args]
    print('$ ' + ' '.join(shlex.quote(arg) for arg in args))
    subprocess.run(args, cwd=str(cwd), check=True, env={**os.environ, **(env or {})})


def index_ready(index_dir: Path) -> bool:
    return all((index_dir / name).exists() for name in ['index.faiss', 'metadata.jsonl', 'config.json'])


display(Markdown(
    f'''**Run config**

- `RUN_MODE`: `{RUN_MODE}`
- `QUESTIONS_CSV`: `{QUESTIONS_CSV}`
- `PDF_DIR`: `{PDF_DIR}`
- `ARTIFACT_DIR`: `{ARTIFACT_DIR}`
- `RESULTS_DIR`: `{RESULTS_DIR}`
- `TOP_K`: `{TOP_K}`
- `FORCE_REBUILD`: `{FORCE_REBUILD}`
    '''
))


In [ ]:
questions_df = pd.read_csv(QUESTIONS_CSV)
required_stems = sorted({Path(token).stem for value in questions_df['source_files'].fillna('') for token in str(value).split() if token})
existing_pdf_count = sum((PDF_DIR / f'{stem}.pdf').exists() for stem in required_stems)

print(f'Questions: {len(questions_df)}')
print(f'Unique required PDFs: {len(required_stems)}')
print(f'PDFs already present in Drive: {existing_pdf_count}')

display(questions_df[['uid', 'difficulty', 'question', 'source_files', 'source_docs']].head(10))


In [ ]:
if DOWNLOAD_MISSING_PDFS:
    run_cmd([
        sys.executable,
        'scripts/download_officeqa_pdfs.py',
        '--questions-csv', QUESTIONS_CSV,
        '--pdf-dir', PDF_DIR,
        '--skip-existing',
    ])

if FORCE_REBUILD or not MANIFEST_PATH.exists():
    run_cmd([
        sys.executable,
        'scripts/prepare_data.py',
        '--questions-csv', QUESTIONS_CSV,
        '--pdf-dir', PDF_DIR,
        '--manifest-out', MANIFEST_PATH,
        '--sanity-out', SANITY_PATH,
    ])
else:
    print(f'Skipping manifest build because {MANIFEST_PATH} already exists.')

if FORCE_REBUILD or not BM25_INDEX_PATH.exists():
    run_cmd([
        sys.executable,
        'scripts/build_page_index.py',
        '--manifest', MANIFEST_PATH,
        '--index-out', BM25_INDEX_PATH,
    ])
else:
    print(f'Skipping BM25 build because {BM25_INDEX_PATH} already exists.')


In [ ]:
for experiment in EXPERIMENTS:
    if experiment['eval_model'] == 'bm25':
        continue

    index_dir = experiment['index_path']
    if not FORCE_REBUILD and index_ready(index_dir):
        print(f'Skipping {experiment["run_name"]} index because it already exists at {index_dir}.')
        continue

    run_cmd([
        sys.executable,
        'scripts/build_multimodal_index.py',
        '--manifest', MANIFEST_PATH,
        '--index-dir', index_dir,
        '--model', experiment['index_model'],
        '--crop-mode', experiment['crop_mode'],
        '--render-cache', RENDER_CACHE,
        '--batch-size', str(BATCH_SIZE),
        '--page-batch-size', str(PAGE_BATCH_SIZE),
        '--dpi', str(DPI),
    ])


In [ ]:
for experiment in EXPERIMENTS:
    out_dir = RESULTS_DIR / experiment['run_name']
    metrics_path = out_dir / 'metrics.json'
    if metrics_path.exists() and not FORCE_REBUILD:
        print(f'Skipping {experiment["run_name"]} eval because {metrics_path} already exists.')
        continue

    run_cmd([
        sys.executable,
        'scripts/run_retrieval_eval.py',
        '--questions-csv', QUESTIONS_CSV,
        '--index', experiment['index_path'],
        '--out-dir', out_dir,
        '--model', experiment['eval_model'],
        '--top-k', str(TOP_K),
    ])

run_cmd([
    sys.executable,
    'scripts/make_report_tables.py',
    '--results-dir', RESULTS_DIR,
    '--summary-out', SUMMARY_PATH,
])

summary_df = pd.read_csv(SUMMARY_PATH).sort_values('page_mrr', ascending=False)
display(summary_df)


## Analysis Exports

The next cells create paper-friendly CSVs and plots under:

`/content/drive/MyDrive/officeqa/results/<RUN_TAG>/analysis/`


In [ ]:
import matplotlib.pyplot as plt

summary_df = pd.read_csv(SUMMARY_PATH).sort_values('page_mrr', ascending=False)
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
summary_df.to_csv(ANALYSIS_DIR / 'summary_table.csv', index=False)

page_cols = ['page_mrr', 'page_recall_at_1', 'page_recall_at_5', 'page_recall_at_10']
doc_cols = ['doc_mrr', 'doc_recall_at_1', 'doc_recall_at_5', 'doc_recall_at_10']

page_plot_df = summary_df.set_index('run_name')[page_cols]
doc_plot_df = summary_df.set_index('run_name')[doc_cols]

ax = page_plot_df.plot(kind='bar', figsize=(12, 5), rot=30, title='OfficeQA Page Retrieval Metrics by Run')
ax.set_ylabel('score')
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.savefig(ANALYSIS_DIR / 'summary_page_metrics.png', dpi=200, bbox_inches='tight')
plt.show()
plt.close()

ax = doc_plot_df.plot(kind='bar', figsize=(12, 5), rot=30, title='OfficeQA Document Retrieval Metrics by Run')
ax.set_ylabel('score')
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.savefig(ANALYSIS_DIR / 'summary_doc_metrics.png', dpi=200, bbox_inches='tight')
plt.show()
plt.close()

print(f'Saved summary table to {ANALYSIS_DIR / "summary_table.csv"}')
print(f'Saved page-metric plot to {ANALYSIS_DIR / "summary_page_metrics.png"}')
print(f'Saved doc-metric plot to {ANALYSIS_DIR / "summary_doc_metrics.png"}')


In [ ]:
import json

records = []
for metrics_path in sorted(RESULTS_DIR.glob('*/metrics.json')):
    data = json.loads(metrics_path.read_text())
    run_name = metrics_path.parent.name
    for row in data['per_query']:
        item = dict(row)
        item['run_name'] = run_name
        records.append(item)

per_query_df = pd.DataFrame(records)
per_query_df.to_csv(ANALYSIS_DIR / 'per_query_metrics.csv', index=False)
display(per_query_df.head(20))

pivot = per_query_df.pivot(index='uid', columns='run_name', values='page_mrr').fillna(0).sort_index()
display(pivot.head(20))

if len(pivot) <= 30:
    ax = pivot.plot(kind='bar', figsize=(14, 6), rot=45, title='Per-question Page MRR by Run')
    ax.set_ylabel('page MRR')
    plt.tight_layout()
    plt.savefig(ANALYSIS_DIR / 'per_query_page_mrr.png', dpi=200, bbox_inches='tight')
    plt.show()
    plt.close()
else:
    print('Skipping per-question bar chart because there are more than 30 questions. Use the CSV export instead.')

best_run_by_query = per_query_df.sort_values(['uid', 'page_mrr', 'doc_mrr'], ascending=[True, False, False]).drop_duplicates('uid')
best_run_by_query.to_csv(ANALYSIS_DIR / 'best_run_by_query.csv', index=False)

difficulty_summary = (
    per_query_df.groupby(['run_name', 'difficulty'])[['page_mrr', 'doc_mrr', 'page_recall_at_10', 'doc_recall_at_10']]
    .mean()
    .reset_index()
)
difficulty_summary.to_csv(ANALYSIS_DIR / 'difficulty_summary.csv', index=False)
display(difficulty_summary)

print(f'Saved per-query CSV to {ANALYSIS_DIR / "per_query_metrics.csv"}')
print(f'Saved best-run-by-query CSV to {ANALYSIS_DIR / "best_run_by_query.csv"}')
print(f'Saved difficulty summary CSV to {ANALYSIS_DIR / "difficulty_summary.csv"}')


In [ ]:
import json
from pathlib import Path
from PIL import Image
from IPython.display import display

TARGET_UID = questions_df['uid'].iloc[0]
TARGET_RUN = 'clip_faiss_fixed'
TOP_N = 3

question_row = questions_df.loc[questions_df['uid'] == TARGET_UID].iloc[0]
print('Question UID:', TARGET_UID)
print('Question:', question_row['question'])
print('Gold source_files:', question_row['source_files'])
print('Gold source_docs:', question_row['source_docs'])

rows = []
for ranked_path in sorted(RESULTS_DIR.glob('*/ranked_pages.jsonl')):
    run_name = ranked_path.parent.name
    with ranked_path.open('r', encoding='utf-8') as handle:
        for line in handle:
            row = json.loads(line)
            if row['uid'] == TARGET_UID and row['rank'] <= 10:
                row['run_name'] = run_name
                rows.append(row)

qual_df = pd.DataFrame(rows).sort_values(['run_name', 'rank'])
qual_path = ANALYSIS_DIR / f'qualitative_{TARGET_UID}.csv'
qual_df.to_csv(qual_path, index=False)
display(qual_df[['run_name', 'rank', 'doc_id', 'page_num', 'score', 'bm25_score', 'matched_crop']])
print(f'Saved qualitative CSV to {qual_path}')

selected = qual_df.loc[qual_df['run_name'] == TARGET_RUN].head(TOP_N)
for _, row in selected.iterrows():
    page_num = int(row['page_num'])
    page_image_path = RENDER_CACHE / row['doc_id'] / f'page_{page_num:04d}.png'
    print(f"\n{TARGET_RUN} rank={row['rank']} doc={row['doc_id']} page={page_num} score={row['score']:.4f}")
    if page_image_path.exists():
        display(Image.open(page_image_path))
    matched_crop = row.get('matched_crop')
    if isinstance(matched_crop, str) and matched_crop:
        crop_path = RENDER_CACHE / row['doc_id'] / f'page_{page_num:04d}_crops' / matched_crop
        if crop_path.exists():
            print(f'Matched crop: {matched_crop}')
            display(Image.open(crop_path))
